In [ ]:
from pathlib import Path


import pandas as pd
import numpy as np

from five_safes_tes_workbench.workbench import Workbench
from partialstats.partials import SumOfSquaresPartial
from partialstats.combiners import mean_combiner, variance_combiner
from partialstats.combiners.scalar import sum_combiner

In [ ]:
wb = Workbench()

wb.validate(config_path="config.yaml") #type: ignore

In [ ]:
sys_pressure_neoplasm_query = """
WITH last_occurrence AS (
    SELECT
        person_id,
        value_as_number,
        ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY measurement_datetime DESC NULLS LAST
        ) AS rn
    FROM "DelphiDemo".measurement
    WHERE measurement_concept_id = 3004249
      AND value_as_number IS NOT NULL
),

value_with_status AS (
    SELECT
        value_as_number,
        CASE
            WHEN person_id IN (
                SELECT person_id
                FROM "DelphiDemo".condition_occurrence
                WHERE condition_concept_id = 139750
            )
            THEN 'with'
            ELSE 'without'
        END AS condition_status
    FROM last_occurrence
    WHERE rn = 1
)

SELECT
    condition_status,
    COUNT(value_as_number) AS count,
    SUM(value_as_number) AS sum,
    SUM(value_as_number * value_as_number) AS sumsq
FROM value_with_status
GROUP BY condition_status;
"""

wb.build_tes.simple_sql(
    name="Mean and variance systolic blood pressure",
    query=sys_pressure_neoplasm_query
)

wb.submit()

**Make sure to update the path of the output**

- Your path id will be different

In [ ]:
wb.fetch_outputs()

In [ ]:
data_paths = [
    "./output/Nottingham TRE 01/1199/output.csv",
    "./output/Nottingham TRE 02/1200/output.csv"
]

In [ ]:
def collect_var_data(
    path: Path,
    group_var: str = "condition_status",
    group_level: str = "with"
) -> SumOfSquaresPartial:
    data = pd.read_csv(path)

    row_data = data[data[group_var] == group_level]

    return SumOfSquaresPartial(
        count=row_data["count"].iloc[0],
        sum=row_data["sum"].iloc[0],
        sumsq=row_data["sumsq"].iloc[0],
    )

In [ ]:
group_levels = ["with", "without"]

summary_rows = []

for group in group_levels:
    group_partials = []

    for path in data_paths:
        partial = collect_var_data(path, group_level=group) #type: ignore
        group_partials.append(partial)

    count = sum_combiner.combine(group_partials)
    mean = mean_combiner.combine(group_partials)
    variance = variance_combiner.combine(group_partials)
    standard_deviation = np.sqrt(variance)

    summary_rows.append({
        "condition_status": group,
        "count": count,
        "mean_systolic_blood_pressure": mean,
        "variance_systolic_blood_pressure": variance,
        "standard_deviation": standard_deviation,
    })

summary_table = pd.DataFrame(summary_rows)

summary_table